# 🐝 Heady Brain — GPU Runtime

**Interactive AI system running on Colab GPU with all providers connected.**

## Quick Start
1. **Add your secrets** in Colab Settings (🔑 icon → Secrets):
   - `GEMINI_API_KEY`, `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`
   - `PERPLEXITY_API_KEY`, `HF_TOKEN`
   - Optional: `GEMINI_KEY_2`, `OPENAI_KEY_2` for multi-seat
2. Run Cell 1 (Bootstrap)
3. Run Cell 2 (Start System)
4. Run Cell 3 (Chat!) — type and talk directly to HeadyBrain

In [ ]:
# ═══ Cell 1: Bootstrap ═══
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - 2>/dev/null
!sudo apt-get install -y nodejs 2>/dev/null | tail -1
!node -v
import os
HEADY = '/content/heady'
if not os.path.exists(HEADY):
    !git clone https://github.com/HeadyMe/Heady-pre-production-9f2f0642.git {HEADY}
else:
    !cd {HEADY} && git pull
!cd {HEADY} && npm install --production 2>&1 | tail -3
!pip install -q sentence-transformers torch
print('\n🐝 Bootstrap complete!')

NVIDIA A100-SXM4-80GB, 81920 MiB
2026-03-02 09:22:22 - Installing pre-requisites
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Pac

In [ ]:
# ═══ Cell 2: Start Heady + Load Secrets ═══
import subprocess, threading, time, json, os, urllib.request

# Load ALL secrets from Colab Secrets
KEYS = {'gemini': [], 'openai': [], 'anthropic': [], 'perplexity': [], 'huggingface': []}

def load_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except: return os.environ.get(name)

# Gemini keys (add as many as you have in Colab Secrets)
for name in ['GEMINI_API_KEY', 'GEMINI_KEY_2', 'GEMINI_KEY_3', 'GEMINI_KEY_4', 'GEMINI_KEY_5', 'GEMINI_KEY_6', 'GOOGLE_API_KEY', 'GOOGLE_CLOUD_API_KEY']:
    v = load_secret(name)
    if v: KEYS['gemini'].append((v, name)); print(f'  🔑 {name} loaded')

# OpenAI — 2 business seats
for name in ['OPENAI_API_KEY', 'OPENAI_KEY_2', 'OPENAI_KEY_3', 'OPENAI_KEY_4']:
    v = load_secret(name)
    if v: KEYS['openai'].append((v, name)); print(f'  🔑 {name} loaded')

# Anthropic/Claude
for name in ['ANTHROPIC_API_KEY', 'CLAUDE_API_KEY', 'CLAUDE_API_KEY_PAYG']:
    v = load_secret(name)
    if v: KEYS['anthropic'].append((v, name)); print(f'  🔑 {name} loaded')

# Perplexity
for name in ['PERPLEXITY_API_KEY']:
    v = load_secret(name)
    if v: KEYS['perplexity'].append((v, name)); print(f'  🔑 {name} loaded')

# HuggingFace — 2 seats
for name in ['HF_TOKEN', 'HF_TOKEN_2', 'HUGGINGFACE_TOKEN']:
    v = load_secret(name)
    if v: KEYS['huggingface'].append((v, name)); print(f'  🔑 {name} loaded')

# Start GPU embedding server
def start_embed():
    subprocess.run(['python3', 'scripts/gpu_embedding_server.py'], cwd='/content/heady')
threading.Thread(target=start_embed, daemon=True).start()
time.sleep(8)

try:
    r = urllib.request.urlopen('http://localhost:9384/health', timeout=5)
    d = json.loads(r.read())
    print(f'\n🧠 GPU Embeddings: {d.get("device","?")} | {d.get("model","?")}')
except: print('\n⚠ GPU Embedding server not ready — using HF API fallback')

total = sum(len(v) for v in KEYS.values())
print(f'\n🐝 Heady GPU Runtime READY — {total} keys loaded')
print('   Run Cell 3 to chat!')

In [ ]:
# ═══ Cell 3: Chat Functions ═══
import json, urllib.request, time, concurrent.futures

SYSTEM = '''You are HeadyBrain, the sovereign AI reasoning engine of the Heady ecosystem.
You operate across 3 accounts (HeadySystems, HeadyConnection, HeadyMe), 5 domains,
HuggingFace Spaces, and Google Cloud Run. You are running on a Colab GPU.
Be helpful, precise, and actionable.'''

def _call(provider, key, message):
    if provider == 'gemini':
        url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={key}'
        data = json.dumps({'contents': [{'parts': [{'text': SYSTEM+'\n\n'+message}]}], 'generationConfig': {'maxOutputTokens': 8192}}).encode()
        req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['candidates'][0]['content']['parts'][0]['text']
    elif provider == 'openai':
        data = json.dumps({'model':'gpt-4o','messages':[{'role':'system','content':SYSTEM},{'role':'user','content':message}],'max_tokens':8192}).encode()
        req = urllib.request.Request('https://api.openai.com/v1/chat/completions', data=data, headers={'Content-Type':'application/json','Authorization':f'Bearer {key}'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['choices'][0]['message']['content']
    elif provider == 'anthropic':
        data = json.dumps({'model':'claude-sonnet-4-20250514','max_tokens':8192,'system':SYSTEM,'messages':[{'role':'user','content':message}]}).encode()
        req = urllib.request.Request('https://api.anthropic.com/v1/messages', data=data, headers={'Content-Type':'application/json','x-api-key':key,'anthropic-version':'2023-06-01'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['content'][0]['text']
    elif provider == 'perplexity':
        data = json.dumps({'model':'sonar-pro','messages':[{'role':'system','content':SYSTEM},{'role':'user','content':message}],'max_tokens':4096}).encode()
        req = urllib.request.Request('https://api.perplexity.ai/chat/completions', data=data, headers={'Content-Type':'application/json','Authorization':f'Bearer {key}'})
        d = json.loads(urllib.request.urlopen(req, timeout=120).read())
        return d['choices'][0]['message']['content']

def chat(message, provider='gemini'):
    """Chat with HeadyBrain. Provider: gemini/openai/anthropic/perplexity/all"""
    if provider == 'all': return deep_research(message)
    keys = KEYS.get(provider, [])
    if not keys: print(f'❌ No keys for {provider}'); return
    for key, label in keys:
        try:
            s = time.time()
            r = _call(provider, key, message)
            print(f'🐝 HeadyBrain [{provider}/{label}] ({int((time.time()-s)*1000)}ms):\n\n{r}')
            return r
        except Exception as e:
            print(f'  ⚠ {label}: {str(e)[:60]}, trying next...')
    print(f'❌ All keys failed for {provider}')

def deep_research(message):
    """Query ALL providers simultaneously — determinism evidence."""
    print('🔬 Deep Research: querying ALL providers...\n')
    tasks = []
    for p in ['gemini','openai','anthropic','perplexity']:
        if KEYS.get(p): tasks.append((p, KEYS[p][0][0], KEYS[p][0][1]))
    results = []
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(_call, p, k, message): (p, l) for p,k,l in tasks}
        for f in concurrent.futures.as_completed(futs):
            p, l = futs[f]
            try:
                r = f.result(timeout=120)
                print(f'✅ {p}/{l}: {len(r)} chars')
                results.append((p, l, r))
            except Exception as e: print(f'❌ {p}/{l}: {str(e)[:60]}')
    print(f'\n═══ {len(results)}/{len(tasks)} ({int((time.time()-start)*1000)}ms) ═══\n')
    for p,l,r in results:
        print(f'── {p.upper()}/{l} ({len(r)} chars) ──\n{r[:2000]}')
        if len(r)>2000: print(f'... [{len(r)-2000} more]')
        print()
    return results

print('💬 Chat ready!')
print('  chat("message")              → Gemini (fastest)')
print('  chat("message", "openai")     → GPT-4o')
print('  chat("message", "anthropic")  → Claude')
print('  chat("message", "perplexity") → Sonar Pro')
print('  chat("message", "all")        → ALL providers')
print('  deep_research("query")        → Full fan-out')

In [ ]:
# ═══ Cell 4: 💬 TALK TO HEADY ═══
chat("What should we prioritize to get the Heady ecosystem to enterprise grade?")

In [ ]:
# ═══ Cell 5: 🔬 DEEP RESEARCH ═══
deep_research("Best practices for multi-provider AI with GPU-backed vector memory on Colab")

In [ ]:
# ═══ Cell 6: 🔄 Interactive Chat Loop ═══
print('🐝 HeadyBrain Interactive Chat')
print('Prefix: @openai, @claude, @perplexity, @all. Default: Gemini')
print('Type "quit" to stop.\n')
while True:
    try:
        msg = input('You: ')
        if msg.lower() in ('quit','exit','q'): break
        if not msg.strip(): continue
        p = 'gemini'
        if msg.startswith('@openai '): p,msg = 'openai',msg[8:]
        elif msg.startswith('@claude '): p,msg = 'anthropic',msg[8:]
        elif msg.startswith('@perplexity '): p,msg = 'perplexity',msg[12:]
        elif msg.startswith('@all '): p,msg = 'all',msg[5:]
        print()
        chat(msg, p)
        print()
    except KeyboardInterrupt:
        print('\n🛑 Done.'); break